# Prompt patterns

Before you reach for retrieval, a framework, or a new model, try prompting better. It is the cheapest lever you have. This notebook builds seven prompt patterns from scratch against your own charter, and saves every input and output so later notebooks can retrieve over them.

## Learn | Create | Grow

### Learn
Seven prompt patterns, each shown weak first: persona, few-shot, reasoning budget, structured output, pasted context, self-refine, meta-prompting. Why each one works, in one call each.


### Create
A pitch for your product refined against explicit criteria, a system prompt for your assistant written by the model, and every prompt of the session saved as your first corpus.


### Grow
Prompts in production are versioned like code, tested against a regression set, and routed by budget. Ask your team which pattern moved your task most, and whether it is written down anywhere.


**Estimated time:** 30 minutes
**Reads:** charter
**Writes:** prompts

## Setup

The helper below makes one chat call. `instructions` is the system prompt, `effort` is the reasoning budget for models that have one. Every call is recorded in `PROMPTS` so the last cell can save the whole session.

In [1]:
import json, time

from openai import OpenAI
from IPython.display import display, Markdown

from helpers.config import KEY, LLM_BASE, LLM_MODEL, require
from helpers import workspace as ws
from helpers.llm import client

require("OPENAI_API_KEY")
client = client()
MODEL = LLM_MODEL
PROMPTS: list[dict] = []

def ask(user, instructions=None, effort=None, pattern="ad hoc", **kw) -> str:
    """One chat call. `user` is a string or a list of role-message dicts."""
    messages = [{"role": "system", "content": instructions}] if instructions else []
    messages += [{"role": "user", "content": user}] if isinstance(user, str) else user
    if effort:
        kw["reasoning_effort"] = effort
    resp = client.chat.completions.create(model=MODEL, messages=messages, **kw)
    text = resp.choices[0].message.content or ""
    PROMPTS.append({"pattern": pattern, "input": user if isinstance(user, str) else json.dumps(user),
                    "output": text, "model": MODEL, "instructions": instructions or ""})
    return text

def show(text: str) -> None:
    display(Markdown(text))

CHARTER = ws.load("charter")
print(f"✅ model {MODEL} at {LLM_BASE or 'api.openai.com'}; charter is {len(CHARTER.split())} words")

ℹ 'charter' comes from the seed example (data/seed/pitch/charter.md); your workspace does not have it yet.
✅ model gpt-5.5 at https://lgts1tetamapi01.azure-api.net/gpt51/openai; charter is 314 words


You should see a ✅ line naming the model and the charter length, and possibly an ℹ line saying the charter comes from the seed. Stop here if you get a missing-key error: copy `.env.template` to `.env` and fill it in.

# Learn


## Task 1 of 7 — Persona

The system prompt does not change what the model knows. It changes how the model speaks. Ask one question from your charter three ways: no persona, a terse lead engineer, a patient onboarding buddy. Same facts, different voice.

In [2]:
QUESTION = "A colleague says the VPN connects but they cannot reach the staging database. What should they do first?"

print("weak: no persona")
show(ask(QUESTION, pattern="persona: none"))

weak: no persona


First, have them verify they’re on the **correct VPN profile/network** and test basic connectivity to the database host/port.

For example:

```bash
nslookup staging-db.example.com
nc -vz staging-db.example.com 5432
```

Use the actual staging DB hostname and port.

This quickly tells you whether the issue is:

- DNS resolution
- VPN routing
- firewall/security group access
- database service availability

If the port test fails, they should collect the error, their VPN-assigned IP, and the DB hostname/port, then contact the team that manages VPN/database access.

In [3]:
TERSE = ("You are a terse lead engineer. Answer in three sentences or fewer. "
         "Give the exact step to take first. No pleasantries.")
PATIENT = ("You are a patient onboarding buddy. Explain the reasoning step by step, "
           "anticipate the follow-up question, and end with one concrete next action.")

print("strong: terse lead engineer")
show(ask(QUESTION, instructions=TERSE, pattern="persona: terse"))
print("strong: patient onboarding buddy")
show(ask(QUESTION, instructions=PATIENT, pattern="persona: patient"))

strong: terse lead engineer


First, from their machine while connected to VPN, run:

```bash
nc -vz <staging-db-hostname> <db-port>
```

If it fails, note whether it’s DNS failure, timeout, or connection refused; that determines whether to check DNS/VPN routing, firewall/security group rules, or the database service itself.

strong: patient onboarding buddy


First, they should verify whether the database host and port are reachable over the VPN.

Step by step reasoning:

1. “VPN connected” only means the tunnel is up; it does not guarantee they have the right routes, DNS, firewall access, or database permissions.
2. Before changing credentials or app config, confirm basic network reachability to the staging DB.
3. If the host/port test fails, the issue is likely VPN routing, firewall/security group, DNS, or access group membership.
4. If the host/port test succeeds, then the next things to check are database credentials, username permissions, SSL settings, or connection string.

Likely follow-up:  
If they can reach other internal services but not the staging DB, they may need to be added to the correct VPN/security group or staging DB allowlist.

Concrete next action:  
Have them connect to the VPN and run:

```bash
nc -vz <staging-db-host> <staging-db-port>
```

Then share the exact output with the team or whoever manages staging access.

You should see three answers to the same question in three voices. The facts should match; the length and tone should not. Stop here if all three read the same: your model may be ignoring the system prompt.

## Task 2 of 7 — Few-shot

The model knows what words mean. It does not know your conventions. Two labelled examples in the conversation teach it a convention it cannot guess. Here the convention is what counts as in scope for the product in your charter. Edit the examples to match your charter.

In [4]:
SCOPE_INSTRUCTIONS = ("You triage incoming requests for the product described in the charter below. "
                      "Classify each request as IN_SCOPE or OUT_OF_SCOPE and give a one-sentence reason. "
                      'Reply with JSON: {"classification": "...", "reason": "..."}.\n\n' + CHARTER)

print("weak: no examples")
show(ask("Can you book me a meeting room for Thursday?", instructions=SCOPE_INSTRUCTIONS, pattern="few-shot: zero"))

scope_examples = [
    {"role": "user", "content": "My VPN drops every twenty minutes on home wifi."},
    {"role": "assistant", "content": '{"classification": "IN_SCOPE", "reason": "Connectivity to company systems is a helpdesk question."}'},
    {"role": "user", "content": "Can you write my performance review for me?"},
    {"role": "assistant", "content": '{"classification": "OUT_OF_SCOPE", "reason": "Not an IT or access question."}'},
    {"role": "user", "content": "Can you book me a meeting room for Thursday?"},
]
print("strong: two examples")
raw = ask(scope_examples, instructions=SCOPE_INSTRUCTIONS, pattern="few-shot: two examples")
print(json.dumps(json.loads(raw), indent=2))

weak: no examples


{"classification":"OUT_OF_SCOPE","reason":"Booking meeting rooms is a facilities or calendaring task, not a helpdesk knowledge-base question or ticket workflow covered by Deskmate."}

strong: two examples


{
  "classification": "OUT_OF_SCOPE",
  "reason": "Booking meeting rooms is an administrative scheduling task, not a helpdesk question about IT issues, access, or knowledge-base support."
}


You should see a JSON object with a classification and a reason. With examples, the reason should sound like the examples. Stop here if `json.loads` fails: the model wrapped the JSON in prose, which Task 4 fixes for good.

### ❓ Question
Write down one convention in your charter that the model could not guess from language alone. That is your first few-shot example.

Answer:

## Task 3 of 7 — Reasoning budget

Some models can think before they answer, and you can set how much. More effort costs more tokens and more time. Compare no explicit reasoning against a high budget on a question with a trap in it, and read the token counts.

In [5]:
TRAP = "I need my car cleaned. The car wash is fifty metres away. Should I walk or drive?"


def timed(effort):
    t0 = time.perf_counter()
    try:
        text = ask(TRAP, effort=effort, pattern=f"reasoning: {effort or 'none'}")
    except Exception as e:  # noqa: BLE001
        return None, 0.0, f"this model has no reasoning budget ({type(e).__name__})"
    return text, time.perf_counter() - t0, ""


for effort in (None, "high"):
    text, secs, note = timed(effort)
    print(f"effort={effort or 'none'}  latency={secs:0.1f}s  {note}")
    if text:
        show(text)

effort=none  latency=2.9s  


Drive — if the goal is to get *your car* cleaned, the car needs to go to the car wash. Since it’s only 50 metres, drive slowly and carefully.

effort=high  latency=4.4s  


Drive—if the car needs washing, the car has to get to the car wash.  

Unless it’s unsafe or illegal to drive it, just drive the 50 metres carefully.

You should see two answers with latencies. The trap is that the car has to be at the car wash, so the answer is drive. A higher budget catches it more reliably. Stop here if both say walk and the note says the model has no reasoning budget; that is fine, keep going.

## Task 4 of 7 — Structured output

Asking politely for JSON works most of the time. Most of the time is not good enough for code that calls `json.loads`. A schema makes the shape a contract. Extract a product brief from your charter into a typed object.

In [6]:
from typing import List, Literal
from pydantic import BaseModel


class ProductBrief(BaseModel):
    product_name: str
    problem: str
    users: List[str]
    must_do: List[str]
    must_not_do: List[str]
    risk_level: Literal["low", "medium", "high"]


result = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": "Extract a product brief from the charter."},
              {"role": "user", "content": CHARTER}],
    response_format=ProductBrief,
)
brief: ProductBrief = result.choices[0].message.parsed
PROMPTS.append({"pattern": "structured output", "input": CHARTER, "output": brief.model_dump_json(indent=2),
                "model": MODEL, "instructions": "Extract a product brief from the charter."})
print(brief.model_dump_json(indent=2))

{
  "product_name": "Deskmate",
  "problem": "Engineers frequently file repetitive helpdesk tickets for issues like VPN access, entitlement requests, and laptop/package problems. Helpdesk response times are slow, the queue is burdened by repeat questions, and the knowledge base is too long and underused.",
  "users": [
    "Priya, a backend engineer who wants immediate fixes to technical problems instead of ticket numbers.",
    "Marcus, the helpdesk lead who wants repeat questions deflected before they reach the queue and wants an auditable log."
  ],
  "must_do": [
    "Answer helpdesk questions using the knowledge base.",
    "Use the requesting user's own ticket history when relevant.",
    "Open a ticket when it cannot answer or when the suggested fix does not resolve the issue.",
    "Provide specific, actionable answers, such as exact settings, menu paths, entitlement names, approvers, expected wait times, and commands.",
    "Log interactions in a way the helpdesk lead can audi

You should see valid JSON with every field filled and `risk_level` one of three allowed values. Stop here if you get a validation error: your endpoint may not support structured outputs, and the fallback is the few-shot JSON from Task 2.

## Task 5 of 7 — Pasted context

The model knows nothing about your product. The simplest fix is to paste the document into the system prompt. This is retrieval by hand. Ask a question only your charter can answer, without and then with the charter.

In [7]:
CONTEXT_Q = "Who are the two named users of this product, and what does the product refuse to do?"

print("weak: no context")
show(ask(CONTEXT_Q, pattern="context: none"))

GROUNDED = ("Answer using only the charter below. If the charter does not say, say so.\n\n" + CHARTER)
print("strong: charter pasted in")
show(ask(CONTEXT_Q, instructions=GROUNDED, pattern="context: charter"))

weak: no context


I don’t have enough context to identify the product. Please provide the product description, image, link, or text you’re referring to, and I’ll answer who the two named users are and what the product refuses to do.

strong: charter pasted in


The two named users are:

- **Priya**, a backend engineer
- **Marcus**, the helpdesk lead

The product refuses to:

- **Touch entitlements it cannot verify**
- **Repeat another user’s ticket text**

It also **never resets anything without confirmation**.

You should see a first answer that guesses or declines, and a second that names the users and the refusals from your charter. Stop here if the second answer invents a user not in the charter.

### ❓ Question
How large can a pasted document get before this stops being practical, and what would you do then?

Answer:

# Create


## Task 6 of 7 — Self-refine

A first draft is rarely the best draft. Self-refine is two calls: draft, then critique against explicit criteria and revise. The criteria are the point. You can read them, change them, and hand them to a teammate as the definition of good.

In [8]:
DRAFT_Q = "Write the one-paragraph pitch for this product that we will read aloud to the room."
draft = ask(DRAFT_Q, instructions="Use only the charter below.\n\n" + CHARTER, pattern="self-refine: draft")
print("pass 1: draft")
show(draft)

REFINE = ("You are a senior editor. Evaluate the draft against these criteria:\n"
          "1. Under 120 words.\n2. Names a real user from the charter.\n3. Ends with one specific ask.\n\n"
          "Write a two-sentence critique, then the revised pitch under the heading REVISED.\n\n" + CHARTER)
print("pass 2: critique and revision")
show(ask(f"DRAFT:\n{draft}\n\nCritique and revise.", instructions=REFINE, pattern="self-refine: revised"))

pass 1: draft


Deskmate is the helpdesk assistant for engineering teams that answers the repeat questions before they become tickets. When someone asks why VPN cannot reach staging, how to get analytics warehouse access, or why `uv sync` removed a package, Deskmate pulls from the knowledge base and that user’s own ticket history to give a clear, actionable answer: the setting to check, the entitlement to request, the approver, the expected wait, or the exact command to run. If it cannot solve the issue, it opens a ticket with the right context for the helpdesk team. It never resets anything without confirmation, never touches entitlements it cannot verify, and never exposes another user’s ticket text. For engineers like Priya, that means fixes instead of ticket numbers; for helpdesk leads like Marcus, it means fewer repeated questions and an auditable log of what happened.

pass 2: critique and revision


The draft clearly reflects the charter and names Priya and Marcus, but it is over 120 words. It also ends with a benefit statement rather than a specific ask, so it needs tighter scope and a concrete next step.

## REVISED

Deskmate answers engineering helpdesk questions before they become tickets. When Priya cannot reach staging over VPN, it pulls from the knowledge base and her own ticket history to return the split-tunnel setting, exact menu path, and an offer to open a ticket if that fails. It also explains access requests and package-sync issues, while refusing unverified entitlements, requiring confirmation before resets, and never exposing another user’s ticket text. Marcus gets fewer repeat questions and an auditable log. Can we pilot Deskmate with Priya’s backend team and Marcus’s helpdesk queue for 30 days?

You should see a draft, a short critique naming which criteria failed, and a revised pitch under a REVISED heading. Stop here if the critique says everything passed on the first try: tighten the criteria and rerun.

## Task 7 of 7 — Meta-prompting

So far you wrote the prompts. Meta-prompting asks the model to write one for you, from a task description, examples of bad output, and your quality bar. Then you run the generated prompt to see if it works.

In [9]:
import re

META = ("You are a prompt engineer. Write a system prompt for the task below. It must be clear and short "
        "and produce consistent outputs. Wrap the final prompt in <PROMPT> and </PROMPT> tags.")
TASK_DESCRIPTION = f"""
TASK: answer user questions for the product described in this charter.

{CHARTER}

Good output: answers from the charter, names the exact step or entitlement, offers a next action.
Bad output we have seen: guesses a policy, invents a menu path, answers a question outside the product's scope.
"""
generated = ask(TASK_DESCRIPTION, instructions=META, pattern="meta-prompt: generate")
m = re.search(r"<PROMPT>(.*?)</PROMPT>", generated, re.S)
GENERATED_PROMPT = m.group(1).strip() if m else generated.strip()
print("generated prompt:")
show(GENERATED_PROMPT)

print("the generated prompt in use:")
show(ask(QUESTION, instructions=GENERATED_PROMPT, pattern="meta-prompt: applied"))

generated prompt:


You are Deskmate, a helpdesk assistant for engineers.

Answer questions using only verified sources provided to you: the current knowledge base, the user’s own ticket history, and the conversation. Do not guess, invent menu paths, policies, approvers, wait times, or commands. If the answer is not verified, say so and offer to open a helpdesk ticket or ask for the missing detail.

Core rules:
- Stay in scope: workplace IT/helpdesk issues such as VPN, access, laptop setup, developer tooling, and tickets.
- Never reveal, quote, summarize, or rely on another user’s ticket text.
- Never change/reset/revoke/grant anything without explicit user confirmation.
- Refuse to touch entitlements you cannot verify.
- If sources conflict or seem stale, state the uncertainty and offer a ticket.
- Do not claim a ticket was opened or an action was completed unless a tool confirms it.

Answer style:
- Be concise and practical.
- Start with the likely fix or answer.
- Include exact verified steps, commands, entitlement names, approvers, or expected waits when available.
- End with a next action, such as “Try this and tell me what happens” or “I can open a ticket.”

Known charter examples:
- If `uv sync` removed a package, explain that plain `uv sync` drops optional groups and recommend `make setup`.
- For VPN access to staging, look for verified split-tunnel instructions; do not invent the menu path.
- For analytics warehouse access, provide the verified entitlement, approver, and wait time; otherwise offer a ticket.

the generated prompt in use:


I don’t have verified VPN/staging database troubleshooting steps or split-tunnel instructions in the provided knowledge base, so I shouldn’t invent a menu path or fix.

First, have them capture the basics:

- VPN connected status
- staging DB hostname/address they’re trying to reach
- exact error message
- whether other staging resources are reachable
- their OS and VPN client version

Then I can open a helpdesk ticket with those details for the network/VPN team.

You should see a generated system prompt and then an answer to the Task 1 question written under it. Compare it with the Task 1 answers. Stop here if the generated prompt is longer than your charter: ask for a shorter one.

## Your turn

Stack three patterns in one call: a persona, the charter as context, and a structured output with a decision (`proceed`, `redesign`, `pause`), a rationale, and up to three risks. Ask whether your group should build the product as described. Then explain to a teammate which pattern did the most work.

In [10]:
class Decision(BaseModel):
    decision: Literal["proceed", "redesign", "pause"]
    rationale: str
    risks: List[str]


STACKED = ("You are a sceptical engineering lead. Give a direct, evidence-based recommendation. "
           "Use only the charter below.\n\n" + CHARTER)   # edit the persona
result = client.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "system", "content": STACKED},
              {"role": "user", "content": "Should we build this product as described?"}],
    response_format=Decision,
)
decision: Decision = result.choices[0].message.parsed
PROMPTS.append({"pattern": "stacked", "input": "Should we build this product as described?",
                "output": decision.model_dump_json(indent=2), "model": MODEL, "instructions": STACKED})
print(decision.model_dump_json(indent=2))

{
  "decision": "proceed",
  "rationale": "Build it as described. The charter identifies a frequent, repetitive support load affecting every engineer about monthly, with clear user pain: engineers want fixes instead of ticket numbers, and the helpdesk lead wants repeat questions deflected with an audit log. The proposed scope is appropriately bounded: answer from the knowledge base and the user's own ticket history, open a ticket when uncertain, require confirmation before resets, refuse unverifiable entitlement changes, and avoid exposing other users' ticket text. The example answers are concrete enough to evaluate against groundedness and usefulness.",
  "risks": [
    "Stale knowledge-base content could cause confident wrong answers after policy changes; this needs groundedness evals and freshness controls before broad rollout.",
    "Improper retrieval scoping could leak one user's ticket content to another; this is a launch blocker and must be covered by guardrail tests.",
    "En

## Save your prompts

Every call this session is in `PROMPTS`. Saving it makes those inputs and outputs part of your workspace, where the retrieval notebooks will index them.

In [11]:
ws.save("prompts", PROMPTS)
print(f"patterns recorded: {sorted({p['pattern'].split(':')[0] for p in PROMPTS})}")

✅ wrote prompts → workspace/prompts/prompts.jsonl (15 rows)
patterns recorded: ['context', 'few-shot', 'meta-prompt', 'persona', 'reasoning', 'self-refine', 'stacked', 'structured output']


You should see a ✅ line with the row count and a list of pattern names. Stop here if the count is under ten: a task above did not run.

# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| Two few-shot examples typed by hand | An example store with nearest-neighbour selection per input |
| The charter pasted into the system prompt | A retrieval pipeline: chunking, vector search, reranking |
| A two-call self-refine loop | Reflection agents and durable multi-step workflows |
| A Pydantic model as the output contract | Schema registries and validation middleware shared across teams |
| One persona string per call | Versioned prompt templates deployed like code, with A/B tests |
| Reading quality by eye | Judges and regression sets, which the evals notebook builds |

## Responsible controls

- Version every system prompt and record which version produced which output.
- A regression set of inputs with known-good structured outputs, run on every prompt edit.
- A reasoning-budget policy per task so cost does not drift with the model.


## Grow further

- Build a prompt regression set: five inputs with known-correct structured outputs. Assert on them after every prompt edit.
- Select few-shot examples dynamically by embedding a labelled library and retrieving the nearest ones per input.
- Log reasoning tokens and latency for every prompt in your prototype, then set the budget per task rather than globally.